In [1]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb pypdf sentence-transformers gradio google-generativeai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-monitoring 2.32.

In [2]:
import langchain
import chromadb
import pypdf
import gradio
import sentence_transformers

print("LangChain:", langchain.__version__)
print("ChromaDB:", chromadb.__version__)
print("PyPDF:", pypdf.__version__)
print("Gradio:", gradio.__version__)

print("\nAll core libraries imported successfully!")

LangChain: 1.4.8
ChromaDB: 1.5.9
PyPDF: 7.1.0
Gradio: 6.31.0

All core libraries imported successfully!


In [2]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded PDF files:")
for filename in uploaded.keys():
    print("📄", filename)

Saving Assessment Guidelines.pdf to Assessment Guidelines (1).pdf

Uploaded PDF files:
📄 Assessment Guidelines (1).pdf


In [3]:
import os

pdf_files = [
    filename for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

print("Total PDF documents:", len(pdf_files))

for pdf in pdf_files:
    print("✓", pdf)

Total PDF documents: 1
✓ Assessment Guidelines (1).pdf


In [4]:
from pypdf import PdfReader

documents = []

for filename in pdf_files:
    reader = PdfReader(filename)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text,
                "source": filename,
                "page": page_number
            })

print("PDF extraction completed!")
print("Total extracted pages:", len(documents))

for doc in documents[:3]:
    print("\nSource:", doc["source"])
    print("Page:", doc["page"])
    print("Text preview:", doc["text"][:300])

PDF extraction completed!
Total extracted pages: 1

Source: Assessment Guidelines (1).pdf
Page: 1
Text preview: GENERAL GUIDELINES  
• Test will be conducted in various labs in college campuses  
• You are required to carry a valid govt photo ID to the test venue  
• Ensure you have a comfortable chair and are seated upright.  
• Use of online communication applications during the test is strictly prohibited.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

for doc in documents:
    split_texts = text_splitter.split_text(doc["text"])

    for chunk in split_texts:
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"]
        })

print("Text chunking completed!")
print("Total chunks created:", len(chunks))

print("\nFirst chunk preview:")
print(chunks[0]["text"][:500])
print("\nSource:", chunks[0]["source"])
print("Page:", chunks[0]["page"])

Text chunking completed!
Total chunks created: 3

First chunk preview:
GENERAL GUIDELINES  
• Test will be conducted in various labs in college campuses  
• You are required to carry a valid govt photo ID to the test venue  
• Ensure you have a comfortable chair and are seated upright.  
• Use of online communication applications during the test is strictly prohibited.   
• Close all programs and applications, including communication applications like 
Skype, Microsoft Teams, etc. before you begin to take the test.  
• Mobile phones, calculators, smart watches and 

Source: Assessment Guidelines (1).pdf
Page: 1


In [6]:
!pip install -q langchain-huggingface

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("HuggingFace embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFace embedding model loaded successfully!


In [8]:
sample_text = chunks[0]["text"]

vector = embedding_model.embed_query(sample_text)

print("Embedding generated successfully!")
print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

Embedding generated successfully!
Vector dimensions: 384
First 10 values: [-0.0037524548824876547, 0.014092208817601204, -0.0028960062190890312, -0.06875408440828323, -0.008661854080855846, -0.011786386370658875, -0.026853376999497414, -0.08023998886346817, -0.04718858748674393, -0.017544642090797424]


In [9]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Convert our chunks into LangChain Documents
langchain_docs = [
    Document(
        page_content=chunk["text"],
        metadata={
            "source": chunk["source"],
            "page": chunk["page"]
        }
    )
    for chunk in chunks
]

# Create ChromaDB vector database
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embedding_model,
    collection_name="knowledge_base",
    persist_directory="/content/chroma_db",
    ids=[f"chunk-{i}" for i in range(len(langchain_docs))]
)

print("ChromaDB knowledge base created successfully!")
print("Total stored documents:", vectorstore._collection.count())

ChromaDB knowledge base created successfully!
Total stored documents: 3


In [10]:
query = "What is the main topic discussed in the documents?"

results = vectorstore.similarity_search(
    query,
    k=3
)

print("Search Results:\n")

for i, result in enumerate(results, start=1):
    print(f"Result {i}")
    print("Source:", result.metadata["source"])
    print("Page:", result.metadata["page"])
    print("Content:", result.page_content[:300])
    print("-" * 50)

Search Results:

Result 1
Source: Assessment Guidelines (1).pdf
Page: 1
Content: • You must not use headphones, earphones, or any other type of external audio   
 equipment  
• You must not communicate with any other person by any means  
 
Important note: If any malpractice or violation of instructions is detected during the 
test, the candidate’s application will be cancelled
--------------------------------------------------
Result 2
Source: Assessment Guidelines (1).pdf
Page: 1
Content: GENERAL GUIDELINES  
• Test will be conducted in various labs in college campuses  
• You are required to carry a valid govt photo ID to the test venue  
• Ensure you have a comfortable chair and are seated upright.  
• Use of online communication applications during the test is strictly prohibited.
--------------------------------------------------
Result 3
Source: Assessment Guidelines (1).pdf
Page: 1
Content: window will be treated as a violation of the code of conduct for the test and is   
 aut

In [11]:
!pip install -q -U "langchain-core>=1.7.5" "langchain-google-genai>=4.4.3"

In [12]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("Gemini API key configured!")

Enter your Gemini API key: ··········
Gemini API key configured!


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

print("Gemini 3.5 Flash initialized successfully!")

Gemini 3.5 Flash initialized successfully!


In [17]:
def ask_knowledge_base(question):

    retrieved_docs = vectorstore.similarity_search(
        question,
        k=3
    )

    context = "\n\n".join([
        f"Source: {doc.metadata['source']}, "
        f"Page: {doc.metadata['page']}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    ])

    prompt = f"""
    You are an AI research assistant.

    Answer using ONLY the provided context.

    If the answer is unavailable in the context,
    clearly say the documents do not contain enough information.

    Do not invent facts or citations.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    # Convert Gemini response into plain text
    content = response.content

    if isinstance(content, list):
        answer_parts = []

        for item in content:
            if isinstance(item, dict):
                if item.get("type") == "text":
                    answer_parts.append(item.get("text", ""))
            elif isinstance(item, str):
                answer_parts.append(item)

        answer = "\n".join(answer_parts)

    else:
        answer = str(content)

    return {
        "answer": answer,
        "sources": [
            {
                "source": doc.metadata["source"],
                "page": doc.metadata["page"]
            }
            for doc in retrieved_docs
        ]
    }


In [18]:
result = ask_knowledge_base(
    "What is the main topic discussed in the documents?"
)

print("ANSWER:\n")
print(result["answer"])

print("\nSOURCES:\n")

for source in result["sources"]:
    print(
        f"{source['source']} - Page {source['page']}"
    )

ANSWER:

Based on the provided documents, the main topic is the **general and behavioral guidelines/instructions** for candidates taking a proctored assessment (specifically for the Infosys Recruitment Process). 

This includes rules regarding required identification, prohibited items and applications, test window switching, login timeframes, proctoring, and conduct guidelines.

SOURCES:

Assessment Guidelines (1).pdf - Page 1
Assessment Guidelines (1).pdf - Page 1
Assessment Guidelines (1).pdf - Page 1


In [19]:
import gradio as gr

def chatbot_response(message, history):

    if not message.strip():
        return "Please enter a question."

    try:
        result = ask_knowledge_base(message)

        answer = result["answer"]

        # Remove duplicate source-page combinations
        unique_sources = []
        seen = set()

        for source in result["sources"]:
            key = (source["source"], source["page"])

            if key not in seen:
                seen.add(key)
                unique_sources.append(source)

        # Add citations
        citations = "\n\n### Sources\n"

        for i, source in enumerate(unique_sources, start=1):
            citations += (
                f"\n{i}. **{source['source']}** "
                f"– Page {source['page']}"
            )

        return answer + citations

    except Exception as e:
        return f"Something went wrong: {str(e)}"


demo = gr.ChatInterface(
    fn=chatbot_response,
    title="📚 AI Knowledge Base",
    description=(
        "Ask questions about your uploaded PDF documents. "
        "Get AI-powered answers with source citations."
    ),
    examples=[
        "What is the main topic discussed in the documents?",
        "Summarize the key findings.",
        "What are the important concepts explained?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0689636ed38fbfec1c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
